# 📘 Statistics — Day 23: Logistic Regression
### Readable, step-by-step Jupyter Notebook guide

**Learning goals:** Understand logistic regression, probabilities and thresholds, train a binary classifier, evaluate a confusion matrix and classification metrics, and practice.

**How to use:** Read each explanation, run cells in order, and attempt the exercises before checking the answers.

## 1. What is logistic regression?

Despite its name, logistic regression is commonly used for **classification**. It estimates the probability that an example belongs to a class.

Example: predict whether a message is spam (`1`) or not spam (`0`).

The sigmoid function maps a model score to a value between 0 and 1:

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

This output can be interpreted as the model's estimated probability for class 1.

## 2. Classification threshold

A common default threshold is `0.5`:
- Probability at least 0.5 → predict class 1.
- Probability below 0.5 → predict class 0.

The threshold can be changed based on the cost of false positives and false negatives. The default is not always best.

## 3. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, recall_score
)

print("Libraries imported successfully!")

## 4. Create a sample binary classification dataset

This artificial dataset has two numerical features and a binary target (`0` or `1`). It is for learning only.

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42
)

data = pd.DataFrame(X, columns=["feature_1", "feature_2"])
data["target"] = y
display(data.head())
print("\nClass counts:")
print(data["target"].value_counts().sort_index())

## 5. Visualize the classes

Each point is one example; the color indicates its known class.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data["feature_1"], data["feature_2"], c=data["target"], alpha=0.75)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Example Binary Classification Dataset")
plt.grid(True, alpha=0.25)
plt.show()

## 6. Split the data and train the model

We reserve a test set. The pipeline standardizes features and then fits logistic regression. Scaling is learned from the training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model.fit(X_train, y_train)
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

## 7. Predict labels and probabilities

`predict()` returns class labels. `predict_proba()` returns estimated probabilities for each class. The second column corresponds to class 1 for this binary dataset.

In [ ]:
y_pred = model.predict(X_test)
probability_class_1 = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "actual_class": y_test,
    "predicted_class": y_pred,
    "probability_class_1": np.round(probability_class_1, 3)
})
display(results.head(10))

## 8. Accuracy and confusion matrix

**Accuracy** is the fraction of predictions that are correct.

Confusion matrix terms:
- **True negative (TN):** actual 0, predicted 0
- **False positive (FP):** actual 0, predicted 1
- **False negative (FN):** actual 1, predicted 0
- **True positive (TP):** actual 1, predicted 1

Accuracy can be misleading when classes are imbalanced, so inspect other metrics too.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.show()

## 9. Precision, recall, F1-score, and ROC-AUC

- **Precision:** among predicted positives, the fraction that are truly positive.
- **Recall:** among actual positives, the fraction correctly found.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures ranking ability across thresholds; 0.5 is chance-level ranking in a typical interpretation.

For defect detection, recall may be especially important if missing a real defect is costly.

In [ ]:
print(classification_report(y_test, y_pred, digits=3))
print(f"ROC-AUC: {roc_auc_score(y_test, probability_class_1):.3f}")

## 10. Experiment with a different threshold

A lower threshold usually increases the number of positive predictions and may improve recall while also increasing false positives. Check the trade-off rather than assuming one threshold is best.

In [ ]:
for threshold in [0.25, 0.50, 0.75]:
    y_pred_custom = (probability_class_1 >= threshold).astype(int)
    print(f"\nThreshold: {threshold:.2f}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_custom):.3f}")
    print(f"Recall:   {recall_score(y_test, y_pred_custom):.3f}")

## 11. Practice exercises

1. Is logistic regression commonly used for regression or classification?
2. What range can a sigmoid probability take?
3. What is a false negative?
4. Why might recall matter in a defect-detection system?
5. Compare the metrics for thresholds `0.25`, `0.50`, and `0.75`.

## 12. Suggested answers

1. It is commonly used for classification.
2. Between 0 and 1 (not including the endpoints for finite inputs).
3. An actual positive example incorrectly predicted as negative.
4. High recall can reduce the number of real defects that the system misses.
5. Run the threshold experiment and compare the results. A higher threshold generally produces fewer positive predictions; the exact metric changes depend on the data.

### Day 23 summary

Logistic regression estimates class probabilities. A threshold converts probabilities to labels. Use a confusion matrix and metrics such as precision, recall, F1-score, and ROC-AUC. Choose a threshold based on the real-world cost of errors.